# Reproducing the ML-Only Dangerous FDIA Dataset Report

End-to-end: **(1) data generation → (2) loading → (3) every figure → (4) ML training & testing** (spatial + temporal GNN),
with per-attack-type node-F1 and Boyaci sample-wise F1 (swF1).

> `pip install 'fdia-graph[all]'`. Set `FDIA_GRAPH_TOKEN` to pull the private release, or generate locally as shown.


## 0. Setup


In [ ]:
import os, sys, numpy as np, torch, torch.nn.functional as F, matplotlib.pyplot as plt
import fdia_graph as fg
from fdia_graph.dataset import FAMILIES
sys.path.insert(0, '.')   # so train_gnn.py / train_tgnn.py (next to this notebook) import
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; STEALTHY = {'Ao','ramp','LRA'}
print('families:', FAMILIES, '| device:', DEV)


## 1. Data generation
Benign records are emitted exactly from stored operating states (0-error AC flows); only attacks re-solve a power flow.
Generate a custom IEEE-14 set with research knobs (the full published shards are `fg.load('ieee14'|'ieee118'|'ieee300')`).


In [ ]:
fg.generate('ieee14', name='nb_demo', per_family=1500, n_benign=6000,
            families=['Ao','Ad','As','Ar','ramp','LRA'], attack_intensity=0.15, ramp_rate=0.002, lra_targets=15, seed=123)
print('registered:', 'nb_demo' in fg.list_datasets())


## 2. Load & inspect
`node_x [N,4]=[|V|,P_inj,Q_inj,theta]`, `edge_x [E,2]=[P_from,Q_from]`, availability masks, per-bus label `y`, `family`, `stealthy`, chronological `split`, `seq_id`, `timestep`.


In [ ]:
DS = 'nb_demo'   # or 'ieee14' / 'ieee118' / 'ieee300'
train = fg.load(DS, split='train'); test = fg.load(DS, split='test')
print(train.summary())
b = next(iter(train.loader(batch_size=4)))
print({k: tuple(v.shape) for k,v in b.items() if hasattr(v,'shape')})
arr = test.to_numpy()   # whole split as numpy (also .to_torch()/.to_pandas()/.to_tf())


## 3. Figures
### 3a. Attack-family composition


In [ ]:
fam = arr['family']; fams = [v for k,v in FAMILIES.items() if v!='benign']
counts = [int((fam==k).sum()) for k,v in FAMILIES.items() if v!='benign']
cols = ['#c0392b' if f in STEALTHY else '#2c7fb8' for f in fams]
plt.figure(figsize=(7,3)); plt.bar(fams, counts, color=cols)
plt.title(f'{DS}: attacked records per family (red=stealthy, blue=detectable)'); plt.ylabel('records'); plt.show()


### 3b. The ML-only claim — BDD stealth vs detectability
Ao/ramp/LRA are state-consistent (evade bad-data detection); Ad/As/Ar corrupt measurements (caught).


In [ ]:
import pandapower as pp, pandapower.networks as pn
from pandapower.create import create_measurement
from pandapower.estimation import chi2_analysis
NET = {14: pn.case14, 118: pn.case118, 300: pn.case300}[train.system]; nl = len(NET().line)
SD = dict(pf=.02,qf=.02,v=.005,pi=.03,qi=.03)
def bdd(j):
    nx,nm,ex,em = arr['node_x'][j],arr['node_m'][j],arr['edge_x'][j],arr['edge_m'][j]; est=NET()
    for e in range(nl):
        if em[e,0]: create_measurement(est,'p','line',ex[e,0],max(abs(ex[e,0])*SD['pf'],1e-3),element=e,side='from')
        if em[e,1]: create_measurement(est,'q','line',ex[e,1],max(abs(ex[e,1])*SD['qf'],1e-3),element=e,side='from')
    for bb in range(train.N):
        if nm[bb,1]: create_measurement(est,'p','bus',nx[bb,1],max(abs(nx[bb,1])*SD['pi'],1e-3),element=bb)
        if nm[bb,2]: create_measurement(est,'q','bus',nx[bb,2],max(abs(nx[bb,2])*SD['qi'],1e-3),element=bb)
        if nm[bb,0]: create_measurement(est,'v','bus',nx[bb,0],SD['v'],element=bb)
    try: return not bool(chi2_analysis(est, init='flat'))
    except Exception: return None
rng = np.random.default_rng(0); rates={}
for k,name in FAMILIES.items():
    idx = np.where(fam==k)[0]
    if not len(idx): continue
    v=[bdd(int(j)) for j in rng.choice(idx, min(15,len(idx)), replace=False)]; v=[x for x in v if x is not None]
    rates[name]=100*np.mean(v) if v else 0
plt.figure(figsize=(7,3)); plt.bar(list(rates), list(rates.values()),
    color=['#7f8c8d' if n=='benign' else ('#c0392b' if n in STEALTHY else '#2c7fb8') for n in rates])
plt.axhline(50,ls='--',c='#888'); plt.ylabel('BDD-pass %'); plt.title('Evades classical bad-data detection?'); plt.show(); print(rates)


### 3c. Measurement distributions & attack surface


In [ ]:
ben = fam==0; nx=arr['node_x'][ben]; nm=arr['node_m'][ben]; atk=fam>0; y=arr['y'][atk]
fig,ax=plt.subplots(1,3,figsize=(12,3))
ax[0].hist(nx[:,:,0][nm[:,:,0]>0],bins=50); ax[0].set_title('|V| [p.u.]')
ax[1].hist(y.sum(1),bins=range(0,int(y.sum(1).max())+2)); ax[1].set_title('tampered buses / record')
ax[2].bar(range(train.N), y.mean(0)); ax[2].set_title('per-bus attack probability')
plt.tight_layout(); plt.show()


## 4. ML training & testing — spatial GNN
Edge-conditioned message passing over the measurement graph. Report node-F1 and Boyaci sample-wise F1 (swF1); `pos_weight` counters the sparse-label collapse.


In [ ]:
from train_gnn import EdgeMPNN, node_f1, sample_f1
torch.manual_seed(0); ei = train.edge_index.to(DEV); tl = train.loader(batch_size=128, shuffle=True)
model = EdgeMPNN().to(DEV); opt = torch.optim.Adam(model.parameters(), 1e-3); pw = torch.tensor(8.0, device=DEV)
for ep in range(3):
    model.train()
    for bt in tl:
        lo = model(bt['node_x'].to(DEV),bt['node_m'].to(DEV),bt['edge_x'].to(DEV),bt['edge_m'].to(DEV),ei)
        loss = F.binary_cross_entropy_with_logits(lo, bt['y'].to(DEV), pos_weight=pw)
        opt.zero_grad(); loss.backward(); opt.step()
    print('epoch', ep+1, 'loss', round(loss.item(),3))
model.eval(); L,Y,Fm=[],[],[]
with torch.no_grad():
    for bt in test.loader(batch_size=128, shuffle=False):
        L.append(model(bt['node_x'].to(DEV),bt['node_m'].to(DEV),bt['edge_x'].to(DEV),bt['edge_m'].to(DEV),ei).cpu())
        Y.append(bt['y']); Fm.append(bt['family'])
L,Y,Fm = torch.cat(L),torch.cat(Y),torch.cat(Fm)
print('family   node-F1  swF1')
for k,name in FAMILIES.items():
    m = Fm==k
    if k and m.any(): print(f'  {name:6s}  {node_f1(L,Y,m):.3f}   {sample_f1(L,Y,m):.3f}')


## 5. Stronger baseline — ARMA graph convolution (the report's benchmark model)
Edge-fused ARMA spectral GNN (Bianchi et al., IEEE TPAMI 2021): the branch P/Q flows are fused into buses, then
ARMA rational filters give a broad, adaptive receptive field — much better than a 2-3 hop GCN at localizing the
sparse, long-range attacks on IEEE-118/300. A temporal variant (`train_tgnn.py`, GRU over a time window) is also included.


In [ ]:
# Full ARMA benchmark (as used for the report's benchmark page). GPU-optimized: bf16, GPU-resident data,
# feature standardization, validation-tuned threshold. On the RTX PRO 6000 this is ~2-4 min/system.
#   !python train_arma.py --system ieee118 --epochs 80 --out bench_118.json
from train_arma import ArmaLoc, f1
print('ARMA localizer ready — see train_arma.py for the full training/eval loop.')


## 6. Evaluation protocol
- **Split:** 60/20/20 chronological, cut on sequence boundaries (ramps never straddle).
- **Balance:** equal per attack family; report per-attack-type metrics.
- **Generalization:** `fg.load(DS, split='train', heldout=True)` excludes As/Ar (unseen-attack protocol).
- **Versioning:** `fg.load('ieee118', release='v0.2.0')` pins a dataset version.
